# Hybrid Internalization + TP/SL Backtest with Liquidity and Vol-Scaled ThresholdsThis notebook simulates FX risk internalization with per-lot realized PnL attribution, volatility-based TP/SL, and liquidity constraints.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta
np.random.seed(42)

## Generate mock price & client data

In [ ]:
times = pd.date_range('2025-01-01 08:00', '2025-01-01 17:00', freq='1min')
mid = 1.08 + np.cumsum(np.random.randn(len(times))*0.0002)
bid = mid - 0.0001
ask = mid + 0.0001
price_df = pd.DataFrame({'time':times, 'mid':mid, 'bid':bid, 'ask':ask})

client_times = np.random.choice(times, size=40, replace=False)
client_times.sort()
sizes = np.random.choice([1e6,2e6,3e6,-1e6,-2e6], size=40)
client_trades = pd.DataFrame({
    'time':client_times,
    'size_ccy1':sizes,
    'price':np.interp(pd.to_datetime(client_times).astype(int), pd.to_datetime(price_df['time']).astype(int), price_df['mid'])
})

## Define hybrid strategy with volatility-based TP/SL and liquidity constraints

In [ ]:
class HybridTP_SL_LiqVol:
    def __init__(self, tp_mult=2, sl_mult=2, vol_window=5, max_hedge_per_min=3e6, eod_time='17:00', base_ccy='EUR', quote_ccy='USD'):
        self.tp_mult = tp_mult
        self.sl_mult = sl_mult
        self.vol_window = vol_window
        self.max_hedge_per_min = max_hedge_per_min
        self.eod_time = eod_time
        self.base_ccy = base_ccy
        self.quote_ccy = quote_ccy
        self.inventory_ccy1 = 0.0
        self.cash_ccy1 = 0.0
        self.lot_id = 0
        self.lots = []
        self.lot_archive = []
        self.ledger = []

    def record_trade(self, t, action, px, amt, pnl=0.0):
        self.ledger.append({'time':t, 'action':action, 'price':px, 'amount':amt, 'inventory':self.inventory_ccy1, 'cash':self.cash_ccy1, 'pnl':pnl})

    def compute_vol(self, price_df, t):
        idx = price_df.index[price_df['time']==t][0]
        if idx < self.vol_window:
            return 0.0001
        return np.std(price_df['mid'].iloc[idx-self.vol_window:idx])

    def on_client_trade(self, row, price_df):
        t, px, size = row['time'], row['price'], row['size_ccy1']
        vol = self.compute_vol(price_df, t)
        tp_level = px + np.sign(size)*self.tp_mult*vol
        sl_level = px - np.sign(size)*self.sl_mult*vol
        self.lot_id += 1
        self.lots.append({'id':self.lot_id, 'size':size, 'entry':px, 'tp':tp_level, 'sl':sl_level})
        self.inventory_ccy1 += size
        self.cash_ccy1 -= size*px
        self.record_trade(t, 'CLIENT', px, size)

    def _execute_hedge(self, t, px, hedge_amt, hit_lot_ids):
        pnl = 0
        for lot_id in hit_lot_ids:
            lot = next(l for l in self.lots if l['id']==lot_id)
            realized = -lot['size']*(px - lot['entry'])
            pnl += realized
            self.lot_archive.append({'id':lot['id'], 'entry':lot['entry'], 'exit':px, 'pnl':realized})
        self.lots = [l for l in self.lots if l['id'] not in hit_lot_ids]
        self.inventory_ccy1 -= hedge_amt
        self.cash_ccy1 += hedge_amt*px
        self.record_trade(t, 'HEDGE', px, hedge_amt, pnl)

    def on_time(self, t, price_row):
        mid, bid, ask = price_row['mid'], price_row['bid'], price_row['ask']
        px_check = ask if self.inventory_ccy1>0 else bid
        hit_ids = []
        hedge_amt = 0
        for lot in self.lots:
            if lot['size']>0 and (px_check>=lot['tp'] or px_check<=lot['sl']):
                hedge_amt += lot['size']
                hit_ids.append(lot['id'])
            elif lot['size']<0 and (px_check<=lot['tp'] or px_check>=lot['sl']):
                hedge_amt += lot['size']
                hit_ids.append(lot['id'])
        if abs(hedge_amt)>0:
            hedge_amt = np.sign(hedge_amt)*min(abs(hedge_amt), self.max_hedge_per_min)
            self._execute_hedge(t, mid, hedge_amt, hit_ids)

    def eod_clear(self, t, price):
        if abs(self.inventory_ccy1)>0:
            self._execute_hedge(t, price, self.inventory_ccy1, [lot['id'] for lot in self.lots])

    def run(self, price_df, client_df):
        for _, row in price_df.iterrows():
            t = row['time']
            # client trades
            ct = client_df[client_df['time']==t]
            for _, c in ct.iterrows():
                self.on_client_trade(c, price_df)
            self.on_time(t, row)
            if t.time().strftime('%H:%M')==self.eod_time:
                self.eod_clear(t, row['mid'])
        return pd.DataFrame(self.ledger), pd.DataFrame(self.lot_archive)

## Run single simulation and plot results

In [ ]:
sim = HybridTP_SL_LiqVol(tp_mult=2, sl_mult=2, max_hedge_per_min=3e6)
ledger, lot_archive = sim.run(price_df, client_trades)
ledger['cum_pnl'] = ledger['pnl'].cumsum()
plt.figure(figsize=(10,5))
plt.plot(ledger['time'], ledger['cum_pnl'])
plt.title('Cumulative PnL in EUR')
plt.show()
ledger.tail()

## Parameter sweep example

In [ ]:
results = []
for tp_mult in [1,2,3]:
    for max_hedge in [1e6,3e6,5e6]:
        sim = HybridTP_SL_LiqVol(tp_mult=tp_mult, sl_mult=tp_mult, max_hedge_per_min=max_hedge)
        ledger, _ = sim.run(price_df, client_trades)
        results.append({'tp_mult':tp_mult,'max_hedge':max_hedge,'final_pnl':ledger['pnl'].sum()})
pd.DataFrame(results)